# Multinomial Naive Bayes with TF-IDF

This notebook trains and evaluates Multinomial Naive Bayes using the project's existing question-grouped train, validation, and test split assignments and shared TF-IDF representation.

## Imports and paths

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.naive_bayes import MultinomialNB

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "tfidf"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

expanded = pd.read_csv(
    DATA_DIR / "expanded_dataset.csv",
    usecols=["id", "text", "target"],
)

split_ids = {
    name: set(pd.read_csv(DATA_DIR / f"{name}.csv", usecols=["id"])["id"])
    for name in ("train", "validation", "test")
}

assert not any(
    split_ids[first] & split_ids[second]
    for first, second in (("train", "validation"), ("train", "test"), ("validation", "test"))
)
assert set.union(*split_ids.values()) == set(expanded["id"].unique())

frames = {
    name: expanded[expanded["id"].isin(ids)].copy()
    for name, ids in split_ids.items()
}

print({name: frame.shape for name, frame in frames.items()})
print({name: frame["target"].value_counts().sort_index().to_dict() for name, frame in frames.items()})

## Load or create the shared TF-IDF vectorizer

The vectorizer uses word unigrams and bigrams, with the same settings used by the other classical models. If the shared artifact is absent, it is fit on training answers only.

In [ ]:
vectorizer_path = MODEL_DIR / "tfidf_vectorizer.joblib"

if vectorizer_path.exists():
    vectorizer = joblib.load(vectorizer_path)
else:
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=100_000,
        sublinear_tf=True,
        strip_accents="unicode",
        dtype=np.float32,
    )
    vectorizer.fit(frames["train"]["text"].astype(str))
    joblib.dump(vectorizer, vectorizer_path)

X = {
    name: vectorizer.transform(frame["text"].astype(str))
    for name, frame in frames.items()
}
y = {
    name: frame["target"].astype(int).to_numpy()
    for name, frame in frames.items()
}

print({name: matrix.shape for name, matrix in X.items()})

## Verify Multinomial Naive Bayes compatibility

Multinomial Naive Bayes requires non-negative feature values. TF-IDF values are checked before training.

In [ ]:
negative_values = sum(int((matrix.data < 0).sum()) for matrix in X.values())
print("Negative feature values:", negative_values)
assert negative_values == 0
print("Compatible: all TF-IDF features are non-negative.")

## Select smoothing with validation macro F1

In [ ]:
alpha_candidates = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
validation_results = []

for alpha in alpha_candidates:
    candidate = MultinomialNB(alpha=alpha, fit_prior=True)
    candidate.fit(X["train"], y["train"])
    predictions = candidate.predict(X["validation"])
    validation_results.append({
        "alpha": alpha,
        "accuracy": accuracy_score(y["validation"], predictions),
        "macro_f1": f1_score(y["validation"], predictions, average="macro"),
    })

validation_results = pd.DataFrame(validation_results).sort_values(
    ["macro_f1", "accuracy"], ascending=False
)
display(validation_results.round(4))
best_alpha = float(validation_results.iloc[0]["alpha"])
print(f"Selected alpha={best_alpha:g}")

## Fit final model and evaluate on the held-out test set

In [ ]:
X_train_validation = vstack([X["train"], X["validation"]])
y_train_validation = np.concatenate([y["train"], y["validation"]])

model = MultinomialNB(alpha=best_alpha, fit_prior=True)
model.fit(X_train_validation, y_train_validation)
test_predictions = model.predict(X["test"])

metrics = {
    "accuracy": accuracy_score(y["test"], test_predictions),
    "macro_precision": precision_score(y["test"], test_predictions, average="macro"),
    "macro_recall": recall_score(y["test"], test_predictions, average="macro"),
    "macro_f1": f1_score(y["test"], test_predictions, average="macro"),
}
confusion = confusion_matrix(y["test"], test_predictions, labels=[0, 1])

print(pd.Series(metrics).round(4))
print("Confusion matrix (rows=true, columns=predicted; human, ChatGPT):")
print(confusion)

## Save the model

In [ ]:
model_path = MODEL_DIR / "multinomial_naive_bayes.joblib"
joblib.dump(
    {
        "classifier": model,
        "config": {
            "alpha": best_alpha,
            "fit_prior": True,
            "selection": "validation macro-F1, accuracy tie-breaker",
            "alpha_candidates": alpha_candidates,
            "vectorizer": "tfidf_vectorizer.joblib",
            "feature_compatibility": "TF-IDF values verified non-negative",
        },
        "metrics": metrics,
        "confusion_matrix": confusion.tolist(),
    },
    model_path,
)
print(f"Saved model to {model_path}")